# Chapter 6.4: Overlap Scheduling -- CPU-GPU Overlap

## Learning Objectives

By the end of this notebook, you will:

1. **Understand why CPU-GPU overlap** matters for serving throughput
2. **Learn how scheduling the next batch** while the current batch executes improves utilization
3. **Explore double buffering** strategies for continuous pipeline feeding
4. **Analyze overlap of tokenization, scheduling, and GPU execution**
5. **Trace SGLang's event loop** implementation for overlap scheduling
6. **Measure performance impact** of overlap vs sequential execution
7. **Identify overlap opportunities** in the serving pipeline

## Prerequisites

- Understanding of LLM serving pipeline (prefill, decode)
- Basic knowledge of asynchronous programming
- Familiarity with CPU-GPU interaction model

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hideak1/vllm_study/blob/main/notebooks/part6/chapter_6.4_overlap_scheduling.ipynb)
[![Download Notebook](https://img.shields.io/badge/Download-Notebook-blue)](https://raw.githubusercontent.com/hideak1/vllm_study/main/notebooks/part6/chapter_6.4_overlap_scheduling.ipynb)

**How to do the exercises:**
1. **Google Colab (Recommended)**: Click the "Open in Colab" badge above — you get your own copy with free GPU.
2. **Local Jupyter**: Clone the repo, run `./start.sh`, then open notebooks at `http://localhost:8888`.
3. **Exercises**: Look for cells marked with 🏋️ **Exercise** or **Assignment**. Fill in the `TODO` sections and run the cell to check your work.

> **Tip**: In Colab, the notebook is automatically copied to your Drive — your changes are saved there.

## 1. Why CPU-GPU Overlap Matters

### 1.1 The Problem: Sequential Execution

In a naive serving pipeline, CPU and GPU work sequentially:

```
Sequential (no overlap):

CPU:  [Schedule B1][Wait...][Schedule B2][Wait...][Schedule B3]
GPU:  [Wait..][Execute B1][Wait..][Execute B2][Wait..][Execute B3]
Time: ======================================================>
      ^idle   ^compute   ^idle  ^compute   ^idle  ^compute
      
GPU Utilization: ~60% (40% idle waiting for CPU)
```

### 1.2 The Solution: Overlapped Execution

```
Overlapped:

CPU:  [Sched B1][Sched B2][Sched B3][Sched B4][Sched B5]
GPU:  [Wait][Exec B1][Exec B2][Exec B3][Exec B4][Exec B5]
Time: ============================================>
      ^one-time  ^GPU continuously busy
      
GPU Utilization: ~95%+ (minimal idle time)
```

### 1.3 What Happens During CPU Time?

The CPU does significant work between GPU batches:

| CPU Task | Typical Time | Description |
|----------|-------------|-------------|
| Tokenization | 0.1-1 ms | Convert text to token IDs |
| Scheduling | 0.5-2 ms | Select requests, check cache, build batch |
| Memory management | 0.1-0.5 ms | Allocate/free KV-cache blocks |
| Sampling | 0.1-0.5 ms | Top-k/p, temperature, repetition penalty |
| Detokenization | 0.1-0.5 ms | Convert tokens back to text |
| Network I/O | 0.1-1 ms | Send responses, receive new requests |
| **Total** | **1-5 ms** | |

For a decode step that takes 5-10 ms on GPU, this 1-5 ms CPU overhead is **10-50% overhead** if not overlapped!

In [ ]:
import matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport numpy as np# Consistent color palette for all diagramsBLUE = '#4A90D9'GREEN = '#27AE60'ORANGE = '#F39C12'RED = '#E74C3C'PURPLE = '#8E44AD'LIGHT_BLUE = '#85C1E9'LIGHT_GREEN = '#82E0AA'LIGHT_ORANGE = '#F8C471'LIGHT_RED = '#F1948A'LIGHT_PURPLE = '#C39BD3'GRAY = '#95A5A6'DARK_GRAY = '#2C3E50'fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6))fig.suptitle('CPU-GPU Overlap Scheduling', fontsize=14, fontweight='bold', color='#2C3E50')def draw_gantt(ax, title, bars):    ax.set_xlim(0, 36)    ax.set_ylim(-0.5, 1.5)    ax.set_yticks([0, 1])    ax.set_yticklabels(['GPU', 'CPU'], fontsize=11, fontweight='bold')    ax.set_xlabel('Time (ms)', fontsize=10)    ax.set_title(title, fontsize=12, fontweight='bold', pad=8)    ax.grid(axis='x', alpha=0.3)    for row, start, width, label, color in bars:        ax.barh(row, width, left=start, height=0.6, color=color, alpha=0.85, edgecolor='white', lw=1.5)        if width > 1.5:            ax.text(start + width/2, row, label, ha='center', va='center', fontsize=7,                    color='white', fontweight='bold')# Without overlap (sequential)bars_seq = []t = 0for i in range(4):    bars_seq.append((1, t, 3, f'Sched {i}', BLUE))    t += 3    bars_seq.append((0, t, 8, f'Exec {i}', RED))    t += 8draw_gantt(ax1, 'Without Overlap: CPU idle during GPU, GPU idle during CPU', bars_seq)ax1.text(35, 0.8, f'Total: {t}ms', fontsize=10, fontweight='bold', color=RED, ha='right')# With overlapbars_ovl = []bars_ovl.append((1, 0, 3, 'Sched 0', BLUE))t_gpu = 3t_cpu = 3for i in range(4):    bars_ovl.append((0, t_gpu, 8, f'Exec {i}', RED))    if i < 3:        bars_ovl.append((1, t_cpu, 3, f'Sched {i+1}', BLUE))    t_gpu += 8    t_cpu = t_gpu  # CPU starts when GPU needs next batch    if i < 3:        t_cpu = t_gpu - 5  # overlapdraw_gantt(ax2, 'With Overlap: CPU schedules next batch while GPU executes current', bars_ovl)ax2.text(35, 0.8, f'Total: {t_gpu}ms', fontsize=10, fontweight='bold', color=GREEN, ha='right')plt.tight_layout(rect=[0, 0, 1, 0.93])plt.savefig("cpu_gpu_overlap.png", dpi=150, bbox_inches='tight', facecolor='white')plt.show()

In [ ]:
import time
import threading
from collections import deque
from typing import List, Optional

# Simulate the impact of overlap vs sequential execution

class PipelineSimulator:
    """Simulates LLM serving pipeline with/without overlap."""
    
    def __init__(self, cpu_time_ms=2.0, gpu_time_ms=8.0):
        self.cpu_time = cpu_time_ms
        self.gpu_time = gpu_time_ms
    
    def sequential_execution(self, n_batches):
        """CPU and GPU work sequentially."""
        total_time = 0
        gpu_busy_time = 0
        
        for i in range(n_batches):
            # CPU: schedule batch
            total_time += self.cpu_time
            
            # GPU: execute batch
            total_time += self.gpu_time
            gpu_busy_time += self.gpu_time
        
        return {
            "total_time_ms": total_time,
            "gpu_utilization": gpu_busy_time / total_time,
            "throughput": n_batches / (total_time / 1000),  # batches/sec
        }
    
    def overlapped_execution(self, n_batches):
        """CPU schedules next batch while GPU executes current batch."""
        # First batch: must schedule sequentially
        total_time = self.cpu_time  # Initial scheduling
        gpu_busy_time = 0
        
        for i in range(n_batches):
            # GPU executes current batch while CPU prepares next
            step_time = max(self.gpu_time, self.cpu_time)  # Overlapped!
            total_time += step_time
            gpu_busy_time += self.gpu_time
        
        return {
            "total_time_ms": total_time,
            "gpu_utilization": gpu_busy_time / total_time,
            "throughput": n_batches / (total_time / 1000),
        }

# Run comparison
sim = PipelineSimulator(cpu_time_ms=3.0, gpu_time_ms=8.0)
n_batches = 100

seq = sim.sequential_execution(n_batches)
ovl = sim.overlapped_execution(n_batches)

print("=== Sequential vs Overlapped Execution ===")
print(f"CPU time: {sim.cpu_time} ms, GPU time: {sim.gpu_time} ms")
print(f"Batches: {n_batches}\n")

print(f"{'Metric':<25s} | {'Sequential':>12s} | {'Overlapped':>12s} | {'Improvement':>12s}")
print("-" * 70)
print(f"{'Total time (ms)':<25s} | {seq['total_time_ms']:>11.0f} | {ovl['total_time_ms']:>11.0f} | {seq['total_time_ms']/ovl['total_time_ms']:>11.2f}x")
print(f"{'GPU utilization':<25s} | {seq['gpu_utilization']:>11.1%} | {ovl['gpu_utilization']:>11.1%} | ")
print(f"{'Throughput (batch/s)':<25s} | {seq['throughput']:>11.1f} | {ovl['throughput']:>11.1f} | {ovl['throughput']/seq['throughput']:>11.2f}x")

## 2. SGLang's Overlap Architecture

### 2.1 The Event Loop

SGLang's scheduler runs an event loop that overlaps CPU scheduling with GPU execution:

```
                SGLang Scheduler Event Loop
                
    +---------------------------------------------------+
    |  while True:                                      |
    |    1. recv_requests()    # Get new requests       |
    |    2. process_finished() # Handle completed reqs  |
    |    3. schedule_batch()   # Build next batch       |
    |    4. launch_gpu_work() # Non-blocking GPU submit|
    |    5. (GPU executes in parallel)                  |
    +---------------------------------------------------+
            ^                          ^
            |     CPU does steps 1-3   |  GPU does step 4
            |     for NEXT batch       |  for CURRENT batch
            |     simultaneously!      |
```

### 2.2 Key Insight: Async GPU Launch

CUDA operations are **asynchronous by default**: when you call a CUDA kernel, it returns immediately to the CPU while the GPU processes the work.

```python
# This is NON-BLOCKING:
output = model(input_ids)  # Returns immediately!
# CPU is free to do other work here.
# GPU is still computing.

# Only this is BLOCKING:
torch.cuda.synchronize()  # Waits for GPU to finish
# or
result = output.cpu()     # Copies data, must wait for GPU
```

SGLang exploits this by **never synchronizing** between scheduling and execution unless absolutely necessary.

In [ ]:
# Demonstrate async GPU execution
import torch

if torch.cuda.is_available():
    # Large matrix multiply to create measurable GPU work
    a = torch.randn(4096, 4096, device='cuda')
    b = torch.randn(4096, 4096, device='cuda')
    torch.cuda.synchronize()
    
    # Sequential: wait for each result
    start = time.perf_counter()
    for _ in range(10):
        c = torch.mm(a, b)
        torch.cuda.synchronize()  # BLOCKING: wait for result
    seq_time = (time.perf_counter() - start) * 1000
    
    # Async: launch all, then wait once
    start = time.perf_counter()
    for _ in range(10):
        c = torch.mm(a, b)  # NON-BLOCKING: returns immediately
    torch.cuda.synchronize()  # Wait for all at once
    async_time = (time.perf_counter() - start) * 1000
    
    print("=== Sync vs Async GPU Execution ===")
    print(f"10 matmuls (4096x4096):")
    print(f"  Sync per-op:  {seq_time:.1f} ms (CPU waits after each op)")
    print(f"  Async batch:  {async_time:.1f} ms (CPU launches all, waits once)")
    print(f"  Speedup: {seq_time/async_time:.2f}x")
    print(f"\nNote: The GPU work is the same. The difference is CPU idle time.")
else:
    print("CUDA not available. The key concept:")
    print("  CUDA kernel launches are non-blocking.")
    print("  The CPU can do other work while the GPU computes.")
    print("  Only torch.cuda.synchronize() or data transfer blocks.")

## 3. Double Buffering Strategy

### 3.1 Concept

Double buffering uses two sets of input buffers, alternating between them:

```
Time:  T0        T1        T2        T3        T4
       |         |         |         |         |
CPU:   [Fill A]  [Fill B]  [Fill A]  [Fill B]  [Fill A]
GPU:   [idle]    [Exec A]  [Exec B]  [Exec A]  [Exec B]

Buffer A: [Writing]  [GPU]     [Writing]  [GPU]     [Writing]
Buffer B: [idle]     [Writing] [GPU]      [Writing] [GPU]
```

### 3.2 Implementation Pattern

```python
class DoubleBuffer:
    def __init__(self):
        self.buffers = [Buffer(), Buffer()]  # Two buffers
        self.write_idx = 0  # CPU writes to this buffer
        self.read_idx = 1   # GPU reads from this buffer
    
    def swap(self):
        """Swap read and write buffers."""
        self.write_idx, self.read_idx = self.read_idx, self.write_idx
    
    def get_write_buffer(self):
        return self.buffers[self.write_idx]
    
    def get_read_buffer(self):
        return self.buffers[self.read_idx]
```

### 3.3 SGLang's Approach

SGLang doesn't use explicit double buffering of input tensors. Instead, it uses a more flexible approach:

1. **Build batch metadata** on CPU while GPU is busy
2. **Prepare input tensors** (token IDs, positions) on CPU
3. **Transfer to GPU** (via `tensor.to(device)`, which is fast)
4. **Launch GPU kernels** (non-blocking)
5. Immediately start building the next batch

In [ ]:
# Simulate double buffering pipeline

class DoubleBufferPipeline:
    """Simulated double-buffered inference pipeline."""
    
    def __init__(self, cpu_time_ms, gpu_time_ms, transfer_time_ms=0.5):
        self.cpu_time = cpu_time_ms
        self.gpu_time = gpu_time_ms
        self.transfer_time = transfer_time_ms
    
    def run_sequential(self, n_steps):
        """Sequential: CPU prepare -> Transfer -> GPU execute."""
        total = 0
        for _ in range(n_steps):
            total += self.cpu_time + self.transfer_time + self.gpu_time
        return total
    
    def run_single_buffer(self, n_steps):
        """Single buffer: overlap CPU with GPU, but transfer serialized."""
        total = self.cpu_time + self.transfer_time  # First batch setup
        for _ in range(n_steps):
            # GPU executes while CPU prepares next
            # But transfer must happen between CPU and GPU work
            total += max(self.gpu_time, self.cpu_time) + self.transfer_time
        return total
    
    def run_double_buffer(self, n_steps):
        """Double buffer: overlap CPU + transfer with GPU."""
        total = self.cpu_time + self.transfer_time  # First batch setup
        for _ in range(n_steps):
            # Everything overlaps with GPU execution
            cpu_and_transfer = self.cpu_time + self.transfer_time
            total += max(self.gpu_time, cpu_and_transfer)
        return total

# Compare strategies
pipe = DoubleBufferPipeline(cpu_time_ms=3.0, gpu_time_ms=8.0, transfer_time_ms=0.5)
n_steps = 200

seq_time = pipe.run_sequential(n_steps)
single_time = pipe.run_single_buffer(n_steps)
double_time = pipe.run_double_buffer(n_steps)

print("=== Pipeline Strategy Comparison ===")
print(f"CPU: {pipe.cpu_time}ms, Transfer: {pipe.transfer_time}ms, GPU: {pipe.gpu_time}ms")
print(f"Steps: {n_steps}\n")

print(f"{'Strategy':<20s} | {'Total (ms)':>12s} | {'Per-step (ms)':>13s} | {'Throughput':>12s}")
print("-" * 65)
print(f"{'Sequential':<20s} | {seq_time:>11.0f} | {seq_time/n_steps:>12.1f} | {n_steps/(seq_time/1000):>11.0f}/s")
print(f"{'Single Buffer':<20s} | {single_time:>11.0f} | {single_time/n_steps:>12.1f} | {n_steps/(single_time/1000):>11.0f}/s")
print(f"{'Double Buffer':<20s} | {double_time:>11.0f} | {double_time/n_steps:>12.1f} | {n_steps/(double_time/1000):>11.0f}/s")
print(f"\nDouble buffer speedup over sequential: {seq_time/double_time:.2f}x")

## 4. Detailed Overlap Analysis

### 4.1 The Full Serving Pipeline

Let's break down every step in the serving pipeline and identify overlap opportunities:

```
Full Pipeline Per Step:

1. Receive new requests (Network I/O)        [CPU: 0.1-1 ms]
2. Tokenize new requests                     [CPU: 0.5-2 ms]
3. Check RadixAttention cache                [CPU: 0.1-0.5 ms]
4. Schedule batch (select requests)           [CPU: 0.2-1 ms]
5. Prepare input tensors                      [CPU: 0.1-0.5 ms]
6. Transfer tensors to GPU                    [CPU->GPU: 0.1-0.5 ms]
7. GPU forward pass                           [GPU: 5-50 ms]
8. Sample next token                          [GPU: 0.1-0.5 ms]
9. Transfer results to CPU                    [GPU->CPU: 0.05 ms]
10. Detokenize results                        [CPU: 0.1-0.5 ms]
11. Send streaming responses                  [CPU: 0.1 ms]
12. Update request states                     [CPU: 0.1 ms]
```

### 4.2 Overlap Opportunities

```
Step:     [1-5]  [6] [7.........] [8] [9] [10-12]
Device:   [CPU]  [T] [   GPU    ] [G] [T] [ CPU ]

Overlap:  Steps 10-12 of batch N can overlap with GPU step 7 of batch N+1
          Steps 1-5 of batch N+1 can overlap with GPU step 7 of batch N

Ideal overlap:

Batch N:    [CPU 1-5][T][GPU 7........][G][T]
Batch N+1:               [CPU 1-5][T]      [GPU 7........][G][T]
                         ^^^^^^^^^^         ^^^^^^^^^^^^^^^^^^^
                         These overlap!     GPU continuously busy!
```

In [ ]:
# Detailed pipeline overlap simulation

class DetailedPipeline:
    """Detailed serving pipeline with per-stage timing."""
    
    def __init__(self):
        # Stage timings in ms
        self.stages = {
            "recv_requests":     {"time": 0.3, "device": "cpu"},
            "tokenize":          {"time": 1.0, "device": "cpu"},
            "cache_check":       {"time": 0.3, "device": "cpu"},
            "schedule_batch":    {"time": 0.5, "device": "cpu"},
            "prepare_tensors":   {"time": 0.3, "device": "cpu"},
            "transfer_to_gpu":   {"time": 0.2, "device": "transfer"},
            "gpu_forward":       {"time": 8.0, "device": "gpu"},
            "sampling":          {"time": 0.3, "device": "gpu"},
            "transfer_to_cpu":   {"time": 0.05, "device": "transfer"},
            "detokenize":        {"time": 0.3, "device": "cpu"},
            "send_responses":    {"time": 0.1, "device": "cpu"},
            "update_states":     {"time": 0.1, "device": "cpu"},
        }
    
    def total_cpu_time(self):
        return sum(s["time"] for s in self.stages.values() if s["device"] == "cpu")
    
    def total_gpu_time(self):
        return sum(s["time"] for s in self.stages.values() if s["device"] == "gpu")
    
    def total_transfer_time(self):
        return sum(s["time"] for s in self.stages.values() if s["device"] == "transfer")
    
    def sequential_step_time(self):
        return sum(s["time"] for s in self.stages.values())
    
    def overlapped_step_time(self):
        """With overlap: CPU work for next batch during GPU execution."""
        cpu_time = self.total_cpu_time()
        gpu_time = self.total_gpu_time()
        transfer_time = self.total_transfer_time()
        
        # GPU time dominates; CPU + transfer happens in parallel
        # Only the part that exceeds GPU time adds latency
        return max(gpu_time + transfer_time, cpu_time + transfer_time)

pipe = DetailedPipeline()

print("=== Detailed Pipeline Stage Timing ===")
print(f"\n{'Stage':<25s} | {'Time (ms)':>10s} | {'Device':>8s}")
print("-" * 50)
for name, info in pipe.stages.items():
    print(f"{name:<25s} | {info['time']:>9.2f} | {info['device']:>8s}")

print(f"\n{'='*50}")
print(f"{'Total CPU time:':<25s} {pipe.total_cpu_time():>10.2f} ms")
print(f"{'Total GPU time:':<25s} {pipe.total_gpu_time():>10.2f} ms")
print(f"{'Total transfer time:':<25s} {pipe.total_transfer_time():>10.2f} ms")
print(f"{'Sequential step:':<25s} {pipe.sequential_step_time():>10.2f} ms")
print(f"{'Overlapped step:':<25s} {pipe.overlapped_step_time():>10.2f} ms")
print(f"{'Overlap benefit:':<25s} {pipe.sequential_step_time()/pipe.overlapped_step_time():>10.2f}x")

In [ ]:
# Visualize the pipeline timeline

try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as mpatches
    HAS_PLT = True
except ImportError:
    HAS_PLT = False

if HAS_PLT:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 6))
    
    colors = {"cpu": "#3498db", "gpu": "#e74c3c", "transfer": "#f39c12"}
    
    # Sequential timeline
    t = 0
    for batch in range(3):
        for name, info in pipe.stages.items():
            color = colors[info["device"]]
            row = 0 if info["device"] == "cpu" else (1 if info["device"] == "gpu" else 0.5)
            ax1.barh(row, info["time"], left=t, height=0.4, color=color, alpha=0.7,
                     edgecolor='black', linewidth=0.5)
            t += info["time"]
    
    ax1.set_yticks([0, 0.5, 1])
    ax1.set_yticklabels(['CPU', 'Transfer', 'GPU'])
    ax1.set_title('Sequential Pipeline (No Overlap)')
    ax1.set_xlabel('Time (ms)')
    
    # Overlapped timeline (simplified)
    cpu_tasks = [(n, i) for n, i in pipe.stages.items() if i["device"] in ["cpu", "transfer"]]
    gpu_tasks = [(n, i) for n, i in pipe.stages.items() if i["device"] in ["gpu", "transfer"]]
    
    gpu_start = 0
    for batch in range(3):
        # CPU timeline
        cpu_t = gpu_start  # CPU starts when GPU batch starts
        for name, info in cpu_tasks:
            ax2.barh(0, info["time"], left=cpu_t, height=0.4, 
                     color=colors[info["device"]], alpha=0.7,
                     edgecolor='black', linewidth=0.5)
            cpu_t += info["time"]
        
        # GPU timeline  
        gpu_t = gpu_start + pipe.total_cpu_time() * 0.3  # Some overlap
        for name, info in gpu_tasks:
            ax2.barh(1, info["time"], left=gpu_t, height=0.4,
                     color=colors[info["device"]], alpha=0.7,
                     edgecolor='black', linewidth=0.5)
            gpu_t += info["time"]
        
        gpu_start = gpu_t
    
    ax2.set_yticks([0, 1])
    ax2.set_yticklabels(['CPU', 'GPU'])
    ax2.set_title('Overlapped Pipeline')
    ax2.set_xlabel('Time (ms)')
    
    # Legend
    patches = [mpatches.Patch(color=c, label=l) for l, c in 
               [("CPU", "#3498db"), ("GPU", "#e74c3c"), ("Transfer", "#f39c12")]]
    fig.legend(handles=patches, loc='upper right')
    
    plt.tight_layout()
    plt.show()
else:
    print("Timeline visualization requires matplotlib.")

## 5. SGLang's Event Loop Implementation

### 5.1 The Core Scheduler Loop

```python
# From sglang/srt/managers/scheduler.py

class Scheduler:
    def event_loop_normal(self):
        """Main event loop with overlap scheduling."""
        while True:
            # Phase 1: Process results from PREVIOUS GPU batch
            # (The GPU finished while we were scheduling)
            batch_result = self.tp_worker.get_result()
            if batch_result is not None:
                self.process_batch_result(batch_result)
            
            # Phase 2: Receive new requests from tokenizer
            self.recv_requests()
            
            # Phase 3: Schedule next batch
            # This involves:
            # - Selecting requests from waiting queue
            # - Checking radix cache for prefix matches
            # - Allocating KV-cache memory
            # - Building the batch metadata
            new_batch = self.get_next_batch()
            
            # Phase 4: Submit batch to GPU (NON-BLOCKING)
            if new_batch is not None:
                self.tp_worker.submit_batch(new_batch)
                # Returns immediately!
                # GPU starts working while we loop back to Phase 1
    
    def event_loop_overlap(self):
        """Advanced overlap: schedule while GPU is still running."""
        # Launch first batch
        first_batch = self.get_next_batch()
        self.tp_worker.submit_batch(first_batch)
        
        while True:
            # === These CPU tasks overlap with GPU execution ===
            
            # 1. Process completed results
            self.recv_requests()
            
            # 2. Prepare NEXT batch while GPU runs CURRENT batch
            next_batch = self.get_next_batch()
            
            # === Now we need to wait for GPU ===
            
            # 3. Wait for current GPU batch to complete
            result = self.tp_worker.get_result_blocking()
            self.process_batch_result(result)
            
            # 4. Immediately submit the pre-built next batch
            if next_batch is not None:
                self.tp_worker.submit_batch(next_batch)
```

### 5.2 Key Optimization: Pre-building the Batch

The crucial optimization is that `get_next_batch()` (CPU-intensive) runs **while the GPU is busy** with the current batch. This means:

1. When the GPU finishes, the next batch is **already prepared**
2. We submit it immediately, minimizing GPU idle time
3. The only serialization point is the brief GPU-to-CPU result transfer

In [ ]:
# Simulate SGLang's overlap event loop

class EventLoopSimulator:
    """Simulates SGLang scheduler event loop with timing."""
    
    def __init__(self, schedule_time=2.0, gpu_time=8.0, 
                 recv_time=0.3, process_time=0.5):
        self.schedule_time = schedule_time  # get_next_batch()
        self.gpu_time = gpu_time           # GPU forward pass
        self.recv_time = recv_time         # recv_requests()
        self.process_time = process_time   # process_batch_result()
    
    def run_normal_loop(self, n_steps):
        """Normal loop: sequential phases."""
        total = 0
        timeline = []
        
        for step in range(n_steps):
            step_start = total
            
            # Phase 1: Get GPU result (may need to wait)
            if step > 0:
                total += self.process_time
            
            # Phase 2: Receive new requests
            total += self.recv_time
            
            # Phase 3: Schedule
            total += self.schedule_time
            
            # Phase 4: GPU execution
            gpu_start = total
            total += self.gpu_time
            
            timeline.append({
                "step": step,
                "cpu_end": gpu_start - step_start,
                "total": total - step_start
            })
        
        return total, timeline
    
    def run_overlap_loop(self, n_steps):
        """Overlap loop: CPU work during GPU execution."""
        total = 0
        timeline = []
        
        # First step: no overlap possible
        total += self.recv_time + self.schedule_time
        total += self.gpu_time  # GPU runs
        timeline.append({"step": 0, "cpu_end": self.recv_time + self.schedule_time, 
                         "total": total})
        
        for step in range(1, n_steps):
            step_start = total
            
            # CPU work overlaps with GPU
            cpu_work = self.process_time + self.recv_time + self.schedule_time
            gpu_work = self.gpu_time
            
            # Step time = max(CPU, GPU) since they overlap
            step_time = max(cpu_work, gpu_work)
            total += step_time
            
            timeline.append({
                "step": step,
                "cpu_time": cpu_work,
                "gpu_time": gpu_work,
                "step_time": step_time,
                "overlap_savings": cpu_work + gpu_work - step_time
            })
        
        return total, timeline

sim = EventLoopSimulator(schedule_time=2.0, gpu_time=8.0)
n_steps = 100

normal_time, normal_tl = sim.run_normal_loop(n_steps)
overlap_time, overlap_tl = sim.run_overlap_loop(n_steps)

print("=== Event Loop Simulation ===")
print(f"Schedule: {sim.schedule_time}ms, GPU: {sim.gpu_time}ms")
print(f"Recv: {sim.recv_time}ms, Process: {sim.process_time}ms")
print(f"Steps: {n_steps}\n")

print(f"Normal loop:  {normal_time:.0f} ms total, {normal_time/n_steps:.1f} ms/step")
print(f"Overlap loop: {overlap_time:.0f} ms total, {overlap_time/n_steps:.1f} ms/step")
print(f"Speedup: {normal_time/overlap_time:.2f}x")
print(f"\nOverlap saves {(normal_time-overlap_time)/normal_time:.1%} of total time")

In [ ]:
# Sensitivity analysis: when does overlap matter most?

print("=== Overlap Impact vs CPU/GPU Time Ratio ===")
print(f"{'CPU (ms)':>10s} | {'GPU (ms)':>10s} | {'Ratio':>8s} | {'Speedup':>8s} | {'Comment':>20s}")
print("-" * 65)

test_configs = [
    (0.5, 10.0, "GPU-bound (prefill)"),
    (1.0, 10.0, "Moderate CPU"),
    (2.0, 8.0,  "Balanced"),
    (3.0, 5.0,  "CPU-heavy"),
    (5.0, 5.0,  "Equal CPU/GPU"),
    (8.0, 3.0,  "CPU-bound"),
    (10.0, 2.0, "Very CPU-bound"),
]

for cpu, gpu, comment in test_configs:
    sim = EventLoopSimulator(schedule_time=cpu*0.7, gpu_time=gpu, 
                             recv_time=cpu*0.15, process_time=cpu*0.15)
    normal, _ = sim.run_normal_loop(100)
    overlap, _ = sim.run_overlap_loop(100)
    ratio = cpu / gpu
    speedup = normal / overlap
    print(f"{cpu:>10.1f} | {gpu:>10.1f} | {ratio:>7.2f} | {speedup:>7.2f}x | {comment:>20s}")

print("\nKey insight: Overlap is most beneficial when CPU and GPU times are comparable.")
print("When one dominates, the other is essentially free (already hidden).")

## 6. Advanced: Overlap in Continuous Batching

### 6.1 Continuous Batching Context

In continuous batching, requests can join and leave the batch at any time. This makes overlap scheduling more complex:

```
Batch at step T:     [Req A (decode), Req B (decode), Req C (prefill)]
Batch at step T+1:   [Req A (decode), Req D (prefill)]  # B finished, C done prefill
                                                          # D just arrived
```

The scheduler must:
1. Check which requests finished (from GPU results)
2. Determine which new requests to add
3. Build the new batch metadata

### 6.2 The Dependency Problem

A challenge: to schedule the next batch, we often need to know the results of the current batch:

```
Problem:
  - Request B might finish at step T (depends on EOS token)
  - We can't know until GPU completes step T
  - But we want to schedule step T+1 BEFORE GPU finishes step T!

Solution: Speculative scheduling
  - Assume no requests finish (optimistic)
  - If a request does finish, adjust the batch on the fly
  - Or: schedule multiple potential batches
```

### 6.3 SGLang's Approach

SGLang handles this pragmatically:
1. **Decode batches** are predictable (same requests continue unless EOS)
2. **Prefill requests** can be pre-scheduled (they don't depend on current results)
3. The **minor delay** from waiting for results is acceptable since GPU time dominates

In [ ]:
# Simulate continuous batching with overlap

import random

class ContinuousBatchSimulator:
    """Simulate continuous batching with overlap scheduling."""
    
    def __init__(self, max_batch_size=32, gpu_time_per_token=0.5):
        self.max_batch = max_batch_size
        self.gpu_time_per_token = gpu_time_per_token
        self.schedule_time = 1.5  # ms
    
    def simulate(self, n_requests, overlap=True):
        """Run simulation with arrival pattern."""
        # Generate requests with random output lengths
        requests = []
        for i in range(n_requests):
            requests.append({
                "id": i,
                "arrival": i * 2.0,  # Arrive every 2ms
                "prefill_tokens": random.randint(50, 200),
                "output_tokens": random.randint(10, 100),
                "tokens_generated": 0,
            })
        
        waiting = list(requests)
        running = []
        completed = []
        total_time = 0
        total_gpu_time = 0
        steps = 0
        
        while waiting or running:
            steps += 1
            
            # Add arrived requests
            newly_arrived = [r for r in waiting if r["arrival"] <= total_time]
            for r in newly_arrived:
                waiting.remove(r)
            
            # Schedule: add new requests up to batch limit
            schedule_start = total_time
            while newly_arrived and len(running) < self.max_batch:
                r = newly_arrived.pop(0)
                running.append(r)
            
            if not overlap:
                total_time += self.schedule_time  # Sequential scheduling
            
            if running:
                # GPU: one decode step for all running requests
                gpu_time = len(running) * self.gpu_time_per_token
                
                if overlap:
                    total_time += max(gpu_time, self.schedule_time)
                else:
                    total_time += gpu_time
                
                total_gpu_time += gpu_time
                
                # Check completions
                still_running = []
                for r in running:
                    r["tokens_generated"] += 1
                    if r["tokens_generated"] >= r["output_tokens"]:
                        completed.append(r)
                    else:
                        still_running.append(r)
                running = still_running
            else:
                total_time += 1  # Idle step
            
            if steps > 10000:  # Safety limit
                break
        
        return {
            "total_time": total_time,
            "steps": steps,
            "completed": len(completed),
            "gpu_utilization": total_gpu_time / total_time,
            "throughput": len(completed) / (total_time / 1000),
        }

random.seed(42)
sim = ContinuousBatchSimulator(max_batch_size=16)

print("=== Continuous Batching: Overlap vs Sequential ===")
print(f"Max batch: {sim.max_batch}, GPU time/token: {sim.gpu_time_per_token}ms\n")

for n_req in [20, 50, 100]:
    seq = sim.simulate(n_req, overlap=False)
    ovl = sim.simulate(n_req, overlap=True)
    
    print(f"--- {n_req} requests ---")
    print(f"  Sequential: {seq['total_time']:.0f}ms, "
          f"GPU util={seq['gpu_utilization']:.1%}, "
          f"throughput={seq['throughput']:.1f} req/s")
    print(f"  Overlapped: {ovl['total_time']:.0f}ms, "
          f"GPU util={ovl['gpu_utilization']:.1%}, "
          f"throughput={ovl['throughput']:.1f} req/s")
    print(f"  Speedup: {seq['total_time']/ovl['total_time']:.2f}x\n")

## 7. Source Code Deep Dive

### 7.1 The `event_loop_overlap` Function

```python
# From sglang/srt/managers/scheduler.py

def event_loop_overlap(self):
    """
    The key insight: we divide each iteration into two halves:
    
    First half: GPU executes previous batch
                CPU prepares next batch (OVERLAPPED)
    
    Second half: GPU result ready
                 CPU processes results + submits prepared batch
    """
    result_queue = self.tp_worker.result_queue
    
    # Bootstrap: process first batch
    self.recv_requests()
    batch = self.get_next_batch()
    if batch:
        self.tp_worker.submit(batch)
    
    while True:
        # === OVERLAP PHASE ===
        # While GPU processes current batch:
        # 1. Receive new requests
        self.recv_requests()
        
        # 2. Process any completed requests from previous step
        #    (these results were available from the PREVIOUS iteration)
        self.process_finished_requests()
        
        # 3. Build next batch
        next_batch = self.get_next_batch()
        
        # === SYNC POINT ===
        # Wait for GPU to finish current batch
        result = result_queue.get()  # Blocking wait
        
        # Process GPU results
        self.process_batch_result(result)
        
        # Submit next batch immediately (already prepared!)
        if next_batch:
            self.tp_worker.submit(next_batch)
```

### 7.2 The `tp_worker` Interface

```python
class TpModelWorker:
    """Manages GPU model execution."""
    
    def submit(self, batch):
        """Submit batch for GPU execution (non-blocking)."""
        # Prepare input tensors
        input_ids = batch.input_ids.to(self.device)  # Fast H2D transfer
        positions = batch.positions.to(self.device)
        
        # Launch forward pass (returns immediately)
        with torch.no_grad():
            logits = self.model(input_ids, positions, ...)
        
        # Launch sampling (returns immediately)
        next_tokens = self.sampler(logits)
        
        # Queue result transfer (will complete later)
        self.result_future = next_tokens.cpu(non_blocking=True)
    
    def get_result(self):
        """Get result from previous submission (may block)."""
        torch.cuda.synchronize()  # Wait for GPU
        return self.result_future
```

In [ ]:
# Simulate the full overlap event loop with timing

class OverlapEventLoop:
    """Simulated SGLang overlap event loop with detailed timing."""
    
    def __init__(self):
        self.timings = {
            "recv_requests": 0.3,
            "process_finished": 0.5,
            "get_next_batch": 1.5,
            "submit_batch": 0.2,
            "gpu_forward": 8.0,
            "get_result": 0.1,  # Just the sync overhead
            "process_result": 0.3,
        }
    
    def run(self, n_iterations):
        """Run the overlap event loop."""
        log = []
        total_time = 0
        gpu_busy = 0
        cpu_busy = 0
        
        # Bootstrap
        cpu_work = (
            self.timings["recv_requests"] +
            self.timings["get_next_batch"] +
            self.timings["submit_batch"]
        )
        total_time += cpu_work
        cpu_busy += cpu_work
        
        for i in range(n_iterations):
            # OVERLAP PHASE: CPU and GPU work simultaneously
            overlap_cpu = (
                self.timings["recv_requests"] +
                self.timings["process_finished"] +
                self.timings["get_next_batch"]
            )
            overlap_gpu = self.timings["gpu_forward"]
            
            overlap_time = max(overlap_cpu, overlap_gpu)
            overlap_savings = overlap_cpu + overlap_gpu - overlap_time
            
            # SYNC PHASE: Sequential
            sync_time = (
                self.timings["get_result"] +
                self.timings["process_result"] +
                self.timings["submit_batch"]
            )
            
            step_time = overlap_time + sync_time
            total_time += step_time
            gpu_busy += self.timings["gpu_forward"]
            cpu_busy += overlap_cpu + sync_time
            
            log.append({
                "step": i,
                "overlap_cpu": overlap_cpu,
                "overlap_gpu": overlap_gpu,
                "overlap_savings": overlap_savings,
                "sync_time": sync_time,
                "step_time": step_time,
            })
        
        return {
            "total_time": total_time,
            "gpu_utilization": gpu_busy / total_time,
            "cpu_utilization": cpu_busy / total_time,
            "avg_step_time": total_time / n_iterations,
            "avg_overlap_savings": sum(l["overlap_savings"] for l in log) / len(log),
            "throughput": n_iterations / (total_time / 1000),
            "log": log,
        }

loop = OverlapEventLoop()
result = loop.run(200)

print("=== Overlap Event Loop Results ===")
print(f"\nStage timings:")
for stage, t in loop.timings.items():
    print(f"  {stage:<20s}: {t:.1f} ms")

print(f"\nResults ({200} iterations):")
print(f"  Total time:           {result['total_time']:.0f} ms")
print(f"  Avg step time:        {result['avg_step_time']:.1f} ms")
print(f"  GPU utilization:      {result['gpu_utilization']:.1%}")
print(f"  CPU utilization:      {result['cpu_utilization']:.1%}")
print(f"  Avg overlap savings:  {result['avg_overlap_savings']:.1f} ms/step")
print(f"  Throughput:           {result['throughput']:.1f} iterations/s")

# Compare with no overlap
no_overlap_step = sum(loop.timings.values())
no_overlap_throughput = 1000 / no_overlap_step
print(f"\n  Without overlap:      {no_overlap_step:.1f} ms/step, {no_overlap_throughput:.1f} it/s")
print(f"  Overlap improvement:  {no_overlap_step / result['avg_step_time']:.2f}x")

## 8. Measuring Overlap Efficiency in Practice

### 8.1 CUDA Events for Precise Timing

```python
# Using CUDA events to measure overlap
start_event = torch.cuda.Event(enable_timing=True)
end_event = torch.cuda.Event(enable_timing=True)

start_event.record()  # Record timestamp in GPU stream
# ... GPU work ...
end_event.record()
torch.cuda.synchronize()
gpu_time = start_event.elapsed_time(end_event)  # ms
```

### 8.2 Detecting Overlap Efficiency

```python
# If overlap is working:
total_wall_time ≈ max(cpu_time, gpu_time)

# If overlap is NOT working:
total_wall_time ≈ cpu_time + gpu_time

# Overlap efficiency metric:
overlap_efficiency = 1 - (total_wall_time - max(cpu_time, gpu_time)) / min(cpu_time, gpu_time)
# 1.0 = perfect overlap, 0.0 = no overlap
```

In [ ]:
# Practical overlap measurement

if torch.cuda.is_available():
    # Create workloads
    gpu_tensor = torch.randn(4096, 4096, device='cuda')
    
    def cpu_work(duration_ms=2.0):
        """Simulate CPU work (scheduling, tokenization, etc.)"""
        end_time = time.perf_counter() + duration_ms / 1000
        result = 0
        while time.perf_counter() < end_time:
            result += 1  # Busy wait
        return result
    
    def gpu_work():
        """Launch GPU computation."""
        return torch.mm(gpu_tensor, gpu_tensor)
    
    n_iter = 50
    
    # Test 1: Sequential (CPU then GPU)
    torch.cuda.synchronize()
    start = time.perf_counter()
    for _ in range(n_iter):
        cpu_work(2.0)
        gpu_work()
        torch.cuda.synchronize()
    seq_time = (time.perf_counter() - start) * 1000 / n_iter
    
    # Test 2: Overlapped (GPU runs while CPU works on next)
    torch.cuda.synchronize()
    start = time.perf_counter()
    gpu_work()  # Launch first GPU work
    for _ in range(n_iter):
        cpu_work(2.0)  # CPU work overlaps with GPU
        torch.cuda.synchronize()  # Wait for GPU
        gpu_work()  # Launch next GPU work immediately
    torch.cuda.synchronize()
    ovl_time = (time.perf_counter() - start) * 1000 / n_iter
    
    print("=== Practical Overlap Measurement ===")
    print(f"CPU work: ~2.0 ms")
    print(f"GPU work: matmul 4096x4096\n")
    print(f"Sequential:  {seq_time:.2f} ms/iter")
    print(f"Overlapped:  {ovl_time:.2f} ms/iter")
    print(f"Speedup:     {seq_time/ovl_time:.2f}x")
    
    # Calculate overlap efficiency
    ideal_overlap = max(seq_time - 2.0, 2.0)  # Approximate
    efficiency = (seq_time - ovl_time) / 2.0  # How much of CPU time is hidden
    print(f"Overlap efficiency: {min(efficiency, 1.0):.1%}")
else:
    print("CUDA required for practical overlap measurement.")

## 9. Identifying Overlap Opportunities

### 9.1 Profiling Checklist

To identify overlap opportunities in your serving pipeline:

1. **Profile CPU stages**: Use `time.perf_counter()` or `cProfile`
2. **Profile GPU stages**: Use CUDA events or `torch.profiler`
3. **Check for unnecessary sync points**: `torch.cuda.synchronize()`, `.cpu()`, `.item()`
4. **Measure wall clock vs sum of stages**: If wall clock = sum, no overlap!

### 9.2 Common Anti-patterns

| Anti-pattern | Why It Hurts | Fix |
|-------------|-------------|-----|
| `tensor.item()` in loop | Forces GPU sync | Batch operations |
| `print(tensor)` for debugging | Forces GPU sync | Remove or use async logging |
| CPU-based sampling | Blocks until GPU done | Use GPU sampling |
| Eager detokenization | Blocks for each token | Batch detokenization |
| Memory allocation in hot path | CPU overhead | Pre-allocate pools |

### 9.3 Optimization Strategies

```python
# Bad: Forces sync every iteration
for step in range(n_steps):
    output = model(input_ids)
    next_token = output.argmax(-1).item()  # SYNC! Blocks CPU
    input_ids = torch.tensor([[next_token]])

# Good: Keep everything on GPU
for step in range(n_steps):
    output = model(input_ids)
    next_token = output.argmax(-1, keepdim=True)  # Stays on GPU
    input_ids = next_token  # No sync needed
```

In [ ]:
# Profile a simulated serving pipeline for overlap opportunities

class PipelineProfiler:
    """Profile each stage of the serving pipeline."""
    
    def __init__(self):
        self.stage_times = {}
    
    def time_stage(self, name, fn, *args):
        start = time.perf_counter()
        result = fn(*args)
        elapsed = (time.perf_counter() - start) * 1000
        self.stage_times[name] = elapsed
        return result
    
    def analyze_overlap(self):
        """Analyze overlap potential."""
        cpu_stages = {}
        gpu_stages = {}
        
        gpu_keywords = ["forward", "attention", "matmul", "gpu"]
        
        for name, t in self.stage_times.items():
            if any(k in name.lower() for k in gpu_keywords):
                gpu_stages[name] = t
            else:
                cpu_stages[name] = t
        
        total_cpu = sum(cpu_stages.values())
        total_gpu = sum(gpu_stages.values())
        total_sequential = total_cpu + total_gpu
        total_overlapped = max(total_cpu, total_gpu)
        
        return {
            "cpu_stages": cpu_stages,
            "gpu_stages": gpu_stages,
            "total_cpu": total_cpu,
            "total_gpu": total_gpu,
            "sequential_time": total_sequential,
            "overlapped_time": total_overlapped,
            "overlap_savings": total_sequential - total_overlapped,
            "overlap_potential": (total_sequential - total_overlapped) / total_sequential,
        }

# Simulate profiling
profiler = PipelineProfiler()

# Simulate stage timings
stages = [
    ("tokenization", lambda: time.sleep(0.001)),
    ("cache_lookup", lambda: time.sleep(0.0003)),
    ("scheduling", lambda: time.sleep(0.0005)),
    ("tensor_prep", lambda: time.sleep(0.0003)),
    ("gpu_forward", lambda: time.sleep(0.008)),
    ("gpu_sampling", lambda: time.sleep(0.0003)),
    ("detokenization", lambda: time.sleep(0.0003)),
    ("response_send", lambda: time.sleep(0.0001)),
]

for name, fn in stages:
    profiler.time_stage(name, fn)

analysis = profiler.analyze_overlap()

print("=== Pipeline Overlap Analysis ===")
print(f"\nCPU Stages:")
for name, t in analysis["cpu_stages"].items():
    print(f"  {name:<20s}: {t:.2f} ms")
print(f"  {'TOTAL':<20s}: {analysis['total_cpu']:.2f} ms")

print(f"\nGPU Stages:")
for name, t in analysis["gpu_stages"].items():
    print(f"  {name:<20s}: {t:.2f} ms")
print(f"  {'TOTAL':<20s}: {analysis['total_gpu']:.2f} ms")

print(f"\nOverlap Analysis:")
print(f"  Sequential time:  {analysis['sequential_time']:.2f} ms")
print(f"  Overlapped time:  {analysis['overlapped_time']:.2f} ms")
print(f"  Savings:          {analysis['overlap_savings']:.2f} ms ({analysis['overlap_potential']:.1%})")

## 10. Summary

### Key Takeaways

1. **CPU-GPU overlap** can improve serving throughput by 20-40% by hiding CPU scheduling time behind GPU execution.

2. **CUDA operations are async by default** -- the key is to avoid unnecessary synchronization points (`synchronize()`, `.item()`, `.cpu()`).

3. **SGLang's event loop** pre-builds the next batch while the GPU processes the current batch, then submits it immediately when the GPU finishes.

4. **Double buffering** extends the concept by using two input buffers, allowing CPU data preparation and GPU-CPU transfer to overlap with GPU execution.

5. **Overlap is most beneficial** when CPU and GPU times are comparable. When GPU time dominates (large batches), CPU time is mostly hidden even without optimization.

6. **Profiling is essential** to identify sync points and measure actual overlap efficiency. Use CUDA events and wall-clock comparison.

7. **Common anti-patterns** like `.item()` in loops, print statements, and eager detokenization can destroy overlap and should be eliminated.

## Exercises

### Exercise 1: Measure Overlap Efficiency

Write a profiler that:
1. Instruments each stage of a simulated pipeline
2. Measures wall-clock time with and without overlap
3. Computes the overlap efficiency metric
4. Identifies which stages are bottlenecks

### Exercise 2: Identify Overlap Opportunities

Given the following pipeline code, identify all synchronization points and suggest how to eliminate them:

```python
def serve_batch(model, requests):
    # Stage 1
    tokens = [tokenizer.encode(r.text) for r in requests]
    input_ids = torch.tensor(tokens).cuda()
    
    # Stage 2
    output = model(input_ids)
    
    # Stage 3: SYNC POINT!
    next_tokens = output.argmax(-1).cpu().tolist()
    
    # Stage 4
    for i, tok in enumerate(next_tokens):
        if tok == eos_token:
            requests[i].finished = True
        requests[i].generated.append(tok)
    
    return next_tokens
```

### Exercise 3: Implement Async Pipeline

Build a simple async pipeline using Python's `asyncio` that demonstrates CPU-GPU overlap. The pipeline should process multiple batches and report GPU utilization.

### Exercise 4: Triple Buffering

Extend the double-buffering concept to triple buffering. When would triple buffering be beneficial? Implement and measure the improvement.

In [ ]:
# Exercise 1 Starter Code

class OverlapProfiler:
    """Profile and measure overlap efficiency."""
    
    def __init__(self):
        self.stage_timings = []
    
    def run_sequential(self, cpu_fn, gpu_fn, n_iter=50):
        """Run CPU then GPU sequentially."""
        # YOUR CODE HERE
        pass
    
    def run_overlapped(self, cpu_fn, gpu_fn, n_iter=50):
        """Run CPU and GPU overlapped."""
        # YOUR CODE HERE
        pass
    
    def compute_efficiency(self, sequential_time, overlapped_time, cpu_time, gpu_time):
        """
        Compute overlap efficiency.
        
        Returns 1.0 for perfect overlap, 0.0 for no overlap.
        """
        # YOUR CODE HERE
        ideal_time = max(cpu_time, gpu_time)
        actual_savings = sequential_time - overlapped_time
        max_possible_savings = sequential_time - ideal_time
        
        if max_possible_savings == 0:
            return 1.0
        return actual_savings / max_possible_savings

print("Exercise 1: Implement run_sequential and run_overlapped methods.")

In [ ]:
# Exercise 3 Starter Code: Async Pipeline

import asyncio

class AsyncPipeline:
    """Async pipeline demonstrating CPU-GPU overlap."""
    
    def __init__(self, cpu_time_ms=2.0, gpu_time_ms=8.0):
        self.cpu_time = cpu_time_ms / 1000  # Convert to seconds
        self.gpu_time = gpu_time_ms / 1000
    
    async def cpu_schedule(self, batch_id):
        """Simulate CPU scheduling work."""
        await asyncio.sleep(self.cpu_time)
        return f"batch_{batch_id}_scheduled"
    
    async def gpu_execute(self, batch_data):
        """Simulate GPU execution."""
        await asyncio.sleep(self.gpu_time)
        return f"{batch_data}_executed"
    
    async def run_sequential(self, n_batches):
        """TODO: Implement sequential execution."""
        # YOUR CODE HERE
        pass
    
    async def run_overlapped(self, n_batches):
        """TODO: Implement overlapped execution."""
        # YOUR CODE HERE
        pass

print("Exercise 3: Implement the async pipeline methods.")
print("Use asyncio.gather() for parallel execution of CPU and GPU tasks.")